# Time Series - Collections

This tutorial builds a minimal `TimeSeriesCollection` for one (synthetic) catchment with five sensors that deliberately differ along every axis `TimeSeriesCollection` has to manage:

| Series              | Frequency | Range | Gaps |
|---------------------|---|---|---|
| `P` (rainfall)      | daily | full year | 1 small (3d) + 1 large (20d) |
| `Q1` (streamflow)   | daily | Mar–Nov (shorter) | 1 small (2d) |
| `Q2` (streamflow)   | daily | Mar–Nov (shorter) | 1 small (2d) |
| `T` (air temp.)     | **hourly** | Jan–Jun (shorter, different unit of time) | 1 small (5h) + 1 large (72h) |
| `ET` (evapotransp.) | daily | full year | 2 separate medium gaps (10d, 15d) |

Covered here:

- Building the four series with `TimeSeries.make_synthetic_tsn()`
- Loading them into one `TimeSeriesCollection`
- `standardize()` and what it does to each series
- Cross-series overlap via `get_epochs()`
- Per-series gap/epoch structure via `merge_local_epochs()`

## Notebook setup

For users running this tutorial as a Jupyter Notebook, this cell must be executed first:

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install `plans` in `google.colab`.
# Use `pip install plans` for other environments.

if "google.colab" in sys.modules:
    import os
    os.system(f"{sys.executable} -m pip install -q plans")

from plans.datasets.core import TimeSeries, TimeSeriesCollection

# This avoids warnings related to uninstalled fonts
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

OUTPUT_DIR = Path("outputs/time-series-collection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: ./{OUTPUT_DIR}")

RNG_SEED = 1
np.random.seed(RNG_SEED)

## Build synthetic series

In [ ]:
df_p = TimeSeries.make_synthetic_tsn(
    start="2019-01-01", end="2019-12-31", base=3, trend=0.0,
    amplitude=2, noise_sd=2.5, freq="D",
    seasonal_period="YS", minor_seasonal_period="D", minor_amplitude=0,
    variable="P",
)
df_p["P"] = df_p["P"].clip(lower=0)
df_p.loc[20:22, "P"] = np.nan     # small gap (3d) -> interpolate_gaps should close this
df_p.loc[150:169, "P"] = np.nan   # large gap (20d) -> a real Epoch 0 gap

df_q = TimeSeries.make_synthetic_tsn(
    start="2019-03-01", end="2019-11-30", base=15, trend=0.01,
    amplitude=5, noise_sd=0.3, freq="D",
    seasonal_period="YS", minor_seasonal_period="D", minor_amplitude=0,
    variable="Q",
)
df_q.loc[40:41, "Q"] = np.nan     # small gap (2d)

df_q2 = TimeSeries.make_synthetic_tsn(
    start="2019-03-20", end="2019-11-30", base=15, trend=0.01,
    amplitude=5, noise_sd=0.4, freq="D",
    seasonal_period="YS", minor_seasonal_period="D", minor_amplitude=0,
    variable="Q",
)
df_q2.loc[46:50, "Q"] = np.nan     # small gap (2d)

df_t = TimeSeries.make_synthetic_tsn(
    start="2019-01-01", end="2019-06-30", base=22, trend=0.0,
    amplitude=4, noise_sd=1.5, freq="h",
    seasonal_period="YS", minor_seasonal_period="D", minor_amplitude=3,
    variable="T",
)
df_t.loc[100:104, "T"] = np.nan     # small gap (5h)
df_t.loc[2000:2071, "T"] = np.nan   # large gap (72h)

df_et = TimeSeries.make_synthetic_tsn(
    start="2019-01-01", end="2019-12-31", base=4, trend=0.0,
    amplitude=1.5, noise_sd=0.5, freq="D",
    seasonal_period="YS", minor_seasonal_period="D", minor_amplitude=0,
    variable="ET",
)
df_et["ET"] = df_et["ET"].clip(lower=0)
df_et.loc[60:69, "ET"] = np.nan     # medium gap (10d)
df_et.loc[300:314, "ET"] = np.nan   # medium gap (15d), separate from the first

for df, fname in [(df_p, "rain.csv"), (df_q, "flow.csv"), (df_q2, "flow2.csv"), (df_t, "temp.csv"), (df_et, "et.csv")]:
    df.to_csv(OUTPUT_DIR / fname, index=False, sep=";")

print("5 series written to", OUTPUT_DIR)

## Load into a Time Series Collection

`TimeSeriesCollection` reads one info table describing every series (name, alias, file, variable/datetime columns, units, coordinates):

In [ ]:
info = pd.DataFrame({
    "Name": ["Rain", "Flow", "Flow2", "AirTemp", "ET"],
    "Alias": ["P1", "Q1", "Q2", "T1", "ET1"],
    "File": [str(OUTPUT_DIR / f) for f in ["rain.csv", "flow.csv", "flow2.csv", "temp.csv", "et.csv"]],
    "VarField": ["P", "Q", "Q", "T", "ET"],
    "DtField": ["datetime"] * 5,
    "Units": ["mm", "m3/s", "m3/s", "C", "mm"],
    "X": [0, 0, 0, 0, 0],
    "Y": [0, 0, 0, 0, 0],
    "Code": ["P001", "Q001", "Q002", "T001", "ET001"],
    "Source": ["synthetic"] * 5,
    "Description": [
        "daily rainfall",
        "daily streamflow", "daily streamflow",
        "hourly air temperature",
        "daily evapotranspiration"
    ],
    "Color": ["tab:blue", "tab:purple", "tab:purple", "tab:red", "tab:orange"],
})
info_file = OUTPUT_DIR / "info.csv"
info.to_csv(info_file, index=False, sep=";")

tsc = TimeSeriesCollection(name="MultiSensorCatchment")
tsc.load_data(table_file=str(info_file))
tsc.catalog

## Raw series at a glance

Visualize individual TimeSeries accessing by the name

In [ ]:
tsc.collection["Flow"].view()
tsc.collection["Flow2"].view()

In [ ]:
tsc.collection["Rain"].view()
tsc.collection["AirTemp"].view()
tsc.collection["ET"].view()

## Learn about the epochs

Visualize epochs per Time Series

In [ ]:
tsc.collection["ET"].view_epochs()

In [ ]:
tsc.collection["Rain"].view_epochs()

Learn collection-wise epochs

In [ ]:
df_local_epochs = tsc.merge_local_epochs()
df_local_epochs

Visualize collection-wise epochs

In [ ]:
tsc.view()

Get merged data epochs

In [ ]:
tsc.get_epochs()

## Standardize collection

Regularizes each series onto its own regular time step and closes small gaps (`interpolate_gaps`), then merges all series onto one shared table:

In [ ]:
tsc.standardize()

for name in tsc.collection:
    d = tsc.collection[name]
    print(f"{name:>10}: n={len(d.data):5d}  nan={d.data['v'].isna().sum():5d}  "
          f"freq={d.dtfreq:>3}  start={d.start.date()}  end={d.end.date()}")

## Merge data

The `.merge_data()` is a helper method that makes an outer join for the available time series, returning a dataframe with all the data.

In [ ]:
df_merged = tsc.merge_data()
df_merged

## Recap

- Built 5 synthetic series with independently varied range, gap structure, and time step.
- `TimeSeriesCollection.load_data()` reads them all from one info table into a shared catalog.
- `get_epochs()` / `merge_local_epochs()` give the cross-series and per-series gap/epoch pictures respectively.